# Romania 2001 — national main-allocation acquisition

This notebook **only acquires and checkpoints source records**. It does not calculate sorting, congestion, HERO curves, or scientific results.

The acquisition follows the 41 county units on the Ministry's official 2001 map and saves six report families for each county:

1. originating-school directory;
2. candidate roster with examination score and grades 5–8 average;
3. admitted placements;
4. unassigned applicants;
5. destination-program directory; and
6. program occupancy and observed cutoffs.

The raw archived pages contain names and therefore remain outside Dropbox and Git under `~/Desktop/VECTOR_temp/`. Every successfully parsed page is saved immediately. If the notebook is interrupted, run the acquisition cell again and it will resume from its verified checkpoints.

The second allocation and redundant per-school report views are deliberately excluded from this first national acquisition so that we preserve one coherent main-allocation population.

## 1. Visible settings

These are the settings you are most likely to change. The initial interval is the starting wait between Archive requests. The code speeds up gradually after successful requests and slows down after failures, while staying between the minimum and maximum values.

`RUN_NATIONAL_ACQUISITION = True` means that the acquisition cell will make network requests. Change it to `False` if you only want to inspect saved progress.

In [1]:
# MASTER SWITCH
# True  = the acquisition cell may contact the Internet Archive and save pages.
# False = preview saved checkpoint status only; no archive requests.
RUN_NATIONAL_ACQUISITION = True

# ADAPTIVE PACING
# Start with two seconds between requests. The retriever may move faster after
# sustained success or slower after refusals and network failures.
INITIAL_INTERVAL_SECONDS = 2.0
MINIMUM_INTERVAL_SECONDS = 1.2
MAXIMUM_INTERVAL_SECONDS = 60.0

# SAFE STOP AND RETRIES
# Stop this invocation after ten hours even if the national acquisition has not
# finished. Because every page is checkpointed, rerunning the cell resumes.
MAX_RUNTIME_HOURS = 10.0

# After the first pass, retry incomplete counties twice more. The three-minute
# cooldown gives the Archive a break before each retry pass.
RETRY_PASSES = 3
RETRY_PASS_COOLDOWN_SECONDS = 180.0

print("National acquisition enabled:", RUN_NATIONAL_ACQUISITION)
print(
    "Adaptive request interval:",
    f"{MINIMUM_INTERVAL_SECONDS:.1f}–{MAXIMUM_INTERVAL_SECONDS:.1f} seconds",
    f"(starting at {INITIAL_INTERVAL_SECONDS:.1f})",
)
print("Maximum runtime for this launch:", MAX_RUNTIME_HOURS, "hours")
print("County retry passes:", RETRY_PASSES)

National acquisition enabled: True
Adaptive request interval: 1.2–60.0 seconds (starting at 2.0)
Maximum runtime for this launch: 10.0 hours
County retry passes: 3


## 2. Load the acquisition code

This cell finds the companion Python module from either the repository root or the notebook folder, reloads it so recent edits are visible, and prints the exact private cache location. It makes no network requests.

In [2]:
from pathlib import Path
import importlib
import sys

# Cursor can launch a notebook with several possible working directories. Search
# upward for the education_analysis/code folder instead of assuming one location.
search_bases = [Path.cwd(), *Path.cwd().parents]
code_folder = None
for base in search_bases:
    for candidate in (base / "education_analysis" / "code", base / "code"):
        if (candidate / "EDUCATION_20260930_romania_2001_national_acquisition.py").exists():
            code_folder = candidate.resolve()
            break
    if code_folder is not None:
        break

if code_folder is None:
    raise FileNotFoundError(
        "Could not find education_analysis/code. Open this notebook from the "
        "Cursor Workspace PDE repository and try again."
    )

if str(code_folder) not in sys.path:
    sys.path.insert(0, str(code_folder))

import EDUCATION_20260930_romania_2001_national_acquisition as national_acquisition
import romania_archive_retrieval

# Reload both modules so saving a code correction does not require restarting the kernel.
importlib.reload(romania_archive_retrieval)
national_acquisition = importlib.reload(national_acquisition)

print("Loaded acquisition code from:", code_folder)
print("Private raw-data cache:", national_acquisition.CACHE)
print("Contact identity used in Archive requests: charles.levine@virginia.edu")

Loaded acquisition code from: /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/code
Private raw-data cache: /Users/charleslevine/Desktop/VECTOR_temp/romania_2001_national_main_allocation_v1
Contact identity used in Archive requests: charles.levine@virginia.edu


## 3. Inspect saved progress before downloading

This cell is local and fast. Before the first run, it should report that the county directory is not yet saved. On later runs, it reports how many county units are fully checkpointed.

In [3]:
from pprint import pprint

progress_before = national_acquisition.county_status()
print("Saved national acquisition status before this run:")
pprint(progress_before)

Saved national acquisition status before this run:
{'cache_folder': '/Users/charleslevine/Desktop/VECTOR_temp/romania_2001_national_main_allocation_v1',
 'complete_counties': 0,
 'county_directory_ready': False,
 'county_units': 0,
 'remaining_counties': None}


## 4. Run or resume the national acquisition

**Code and network activity begin when you run the cell below.**

Expected feedback includes:

- the official county-directory count;
- the current county and pass number;
- a checkpoint message after the first page and then every five pages in a large report;
- rows and pages saved for each source family;
- any adaptive slowdown, long wait, parse problem, or unresolved address;
- fixed 404 pages recorded as archive-coverage gaps without falsely slowing the run; and
- the national count of completed counties.

Stopping the cell does not erase completed checkpoints. To resume, run this same cell again. If the ten-hour limit is reached, the code stops safely and you can launch it again.

In [4]:
# This is the only cell that launches the national archive acquisition.
# Leave Cursor and this notebook open while it runs.

try:
    acquisition_result = national_acquisition.acquire(
        live=RUN_NATIONAL_ACQUISITION,
        initial_interval_seconds=INITIAL_INTERVAL_SECONDS,
        minimum_interval_seconds=MINIMUM_INTERVAL_SECONDS,
        maximum_interval_seconds=MAXIMUM_INTERVAL_SECONDS,
        max_runtime_hours=MAX_RUNTIME_HOURS,
        retry_passes=RETRY_PASSES,
        retry_cooldown_seconds=RETRY_PASS_COOLDOWN_SECONDS,
    )
except KeyboardInterrupt:
    print(
        "\nStopped by you. Completed page checkpoints remain saved. "
        "Run this cell again when you want to resume."
    )
    acquisition_result = national_acquisition.county_status()
except national_acquisition.RetrievalStopped as error:
    print(
        "\nThe retriever stopped safely:", error,
        "\nCompleted page checkpoints remain saved. Run this cell again to resume."
    )
    acquisition_result = national_acquisition.county_status()

print("\nStatus at the end of this invocation:")
pprint(acquisition_result)

National acquisition 1/3 — recovering the official county directory.
  Official map recovered: 41 county units.
National acquisition 2/3 — pass 1/3; 41 counties remain.


National pass 1:   0%|          | 0/41 [00:00<?, ?county/s]


County 1/41 this pass: AB — ALBA
    AB origin_school_directory: 1 pages checkpointed, 168 rows so far; 0 discovered pages waiting.
  origin_school_directory: 1 pages, 168 rows
    AB candidate_roster: 1 pages checkpointed, 500 rows so far; 0 discovered pages waiting.
    AB candidate_roster: 5 pages checkpointed, 2,500 rows so far; 0 discovered pages waiting.
  candidate_roster: 6 pages, 2,982 rows
    AB admitted_placements: 1 pages checkpointed, 500 rows so far; 0 discovered pages waiting.
  Unresolved HTTP attempt: None <urlopen error [Errno 61] Connection refused>
  Archive slowed to 2.8s between requests.
  admitted_placements: 1 pages, 500 rows, 1 unresolved
  Unresolved HTTP attempt: None <urlopen error [Errno 61] Connection refused>
  Archive slowed to 5.0s between requests.
  unassigned_applicants: 0 pages, 0 rows, 1 unresolved
  Unresolved HTTP attempt: None <urlopen error [Errno 61] Connection refused>
  Archive slowed to 8.7s between requests.
  program_directory: 0 pages

## 5. Final local audit

Run this cell after the acquisition cell finishes or after you stop it. It does not contact the Archive. A complete acquisition has 41 completed county units and zero remaining. The final manifest exists only after every county report family is complete. Fixed archive gaps remain explicitly recorded for later recovery from redundant Ministry reports.

In [ ]:
final_status = national_acquisition.county_status()
final_manifest = national_acquisition.CACHE / "completed_national_manifest.json"

print("Completed county units:", final_status.get("complete_counties"), "/", final_status.get("county_units"))
print("Remaining county units:", final_status.get("remaining_counties"))
print("Private cache folder:", final_status.get("cache_folder"))
print("Final national manifest present:", final_manifest.exists())
if final_manifest.exists():
    print("Final national manifest:", final_manifest)   
else:
    print("The acquisition is incomplete. Rerun Section 4 to resume.")